#  ⭐ `dim_PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO`


In [2]:
%run ../../config/bootstrap.py


from pathlib import Path
from utils import get_project_root 

import re
import pandas as pd
from unidecode import unidecode

In [3]:
project_root = get_project_root() 

# 🥉 Bronze 

Raw Data
as is production, low quality

In [4]:
path = Path(project_root) / "data/01_bronze/anvisa/Medicamentos/Medicamentos.parquet"
bronze = pd.read_parquet(Path(project_root) / "data/01_bronze/anvisa/Medicamentos/Medicamentos.parquet")  
bronze['PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO']= bronze['PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO'].fillna('DESCONHECIDO')
bronze = (
    bronze[['PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO']]
    .value_counts()
    .rename('FREQUENCIA_REGISTROS')
    .reset_index()
) 
bronze.columns

Index(['PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO', 'FREQUENCIA_REGISTROS'], dtype='object')

In [5]:
bronze.PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO.nunique()

160

In [6]:
bronze.to_csv(Path(project_root) / "eda/Medicamentos/Medicamentos_PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO.csv", sep=";", index=False)

# 🥈 Silver

normalized data, medium quality


### dev

In [8]:
silver = bronze.copy()
silver.columns

Index(['PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO', 'FREQUENCIA_REGISTROS'], dtype='object')

In [9]:
silver.head(2)

,PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO,FREQUENCIA_REGISTROS
0,DESCONHECIDO,534294
1,Erro de Medicação,7969


In [10]:
import re

col = "PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO"

def clean_problema_token(p: str) -> str:
    p = str(p).upper()
    # remove o marcador X000D onde quer que esteja
    p = p.replace("X000D", " ")
    # normaliza separadores (underscore, espaço múltiplo etc.)
    p = re.sub(r"[_\s]+", " ", p)
    p = p.strip()
    # descarta lixo puro
    if p in ("", "X000D"):
        return ""
    return p

# 1) limpar string "bruta"
s = bronze[col].fillna("").astype(str).str.upper()

s = s.str.replace(r"X000D", " ", regex=True)
s = s.str.replace(r'[\r\n\t]', " ", regex=True)
s = s.str.replace('"', "")

# normaliza separador principal como "|"
s = s.str.replace(r"[|_]+", "|", regex=True)
s = s.str.replace(r"\|{2,}", "|", regex=True)
s = s.str.strip(" |")

# 2) virar LISTA já com cada item limpo
bronze[col] = s.apply(
    lambda txt: [
        clean_problema_token(p)
        for p in txt.split("|")
        if clean_problema_token(p)  # só mantém se não for vazio/lixo
    ]
)

In [11]:
bronze[col].head().to_list()


[['DESCONHECIDO'],
 ['ERRO DE MEDICAÇÃO'],
 ['USO OFF-LABEL USO SEM REGISTRO'],
 ['LOTES TESTADOS E DENTRO DAS ESPECIFICAÇÕES'],
 ['USO INCORRETO']]

In [12]:
import re
from unidecode import unidecode
import pandas as pd

def _slug_col_name(texto: str) -> str:
    """
    Normaliza texto para virar nome de coluna:
    - remove acentos
    - coloca maiúsculas
    - troca não-alfanumérico por "_"
    - colapsa múltiplos "_" em um só
    - tira "_" das pontas
    """
    s = unidecode(str(texto)).upper()
    s = re.sub(r"[^A-Z0-9]+", "_", s)
    s = re.sub(r"_+", "_", s)
    s = s.strip("_")
    return s


def expandir_lista_wide(df, col, prefix=None, dtype="int8"):
    """
    Cria colunas dummies (0/1) para cada valor distinto em uma coluna de LISTAS.

    Ex:
      df[col] = ['ABUSO', 'ERRO DE MEDICAÇÃO']
      -> GRUPO_PROB_ABUSO = 1, GRUPO_PROB_ERRO_DE_MEDICACAO = 1

    Parâmetros
    ----------
    df : pd.DataFrame
        DataFrame com a coluna de listas.
    col : str
        Nome da coluna que contém listas (ex: PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO).
    prefix : str, opcional
        Prefixo para os nomes das colunas geradas. Default: f"{col}_" normalizado.
    dtype : str, default "int8"
        Tipo para as colunas 0/1.

    Retorna
    -------
    df : pd.DataFrame
        Mesmo DataFrame de entrada, com novas colunas 0/1.
    """
    if col not in df.columns:
        return df

    # garante prefixo bonitinho
    if prefix is None:
        prefix = _slug_col_name(col) + "_"

    s = df[col]

    # explode para descobrir todos os valores distintos
    categorias = (
        s.explode()
         .dropna()
         .astype(str)
         .str.strip()
         .loc[lambda x: x != ""]
         .drop_duplicates()
         .sort_values()
    )

    for valor in categorias:
        col_out = prefix + _slug_col_name(valor)
        # cria dummy: 1 se o valor estiver na lista daquela linha
        df[col_out] = s.apply(
            lambda lst: valor in lst if isinstance(lst, (list, tuple, set)) else False
        ).astype(dtype)

    return df


In [13]:
bronze = expandir_lista_wide(
    bronze,
    col="PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO",
    prefix="PROB_ADIC_",   # se quiser um prefixo mais curto/bonito
)

bronze.filter(like="PROB_ADIC_").head()


,PROB_ADIC_ABUSO,PROB_ADIC_DESCONHECIDO,PROB_ADIC_ERRO_DE_MEDICACAO,PROB_ADIC_EXPOSICAO_OCUPACIONAL,PROB_ADIC_FALSIFICACAO,PROB_ADIC_LOTES_TESTADOS_E_DENTRO_DAS_ESPECIFICACOES,PROB_ADIC_LOTES_TESTADOS_E_FORA_DAS_ESPECIFICACOES,PROB_ADIC_MEDICAMENTO_TOMADO_FORA_DA_DATA_DE_VALIDADE,PROB_ADIC_MEDICAMENTO_TOMADO_PELO_PAI,PROB_ADIC_SUPERDOSE,PROB_ADIC_USO_INCORRETO,PROB_ADIC_USO_OFF_LABEL_USO_SEM_REGISTRO
0,0,1,0,0,0,0,0,0,0,0,0,0
1,0,0,1,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,0,1
3,0,0,0,0,0,1,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,1,0


## elt

# 🥇 Gold
